# Helmet detector training (YOLO11n)

Trains the *With Helmet / Without Helmet* detector used by the Two-Wheeler Safety project and
evaluates it on the held-out test split.

**Run on Google Colab** with a GPU runtime (*Runtime -> Change runtime type -> T4 GPU*).

1. Download the dataset from Roboflow Universe as **YOLOv8** format (see `data/README.md`).
2. Run the cells top to bottom and upload the dataset zip when asked.
3. Copy the downloaded `best.pt` to `models/helmet_detector.pt` in the project.

## 1. Check the GPU

In [ ]:
import torch

print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Upload and extract the dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import zipfile

# Name of the zip you just uploaded (it appears in /content)
zip_path = "/content/Helmet Detection_YOLOv8.v3-with-and-without-helmet-dataset.yolov8.zip"
extract_path = "/content/helmet_dataset"

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"Upload the dataset zip first, expected: {zip_path}")

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_path)

print("Extracted:", os.listdir(extract_path))
for split in ("train", "valid", "test"):
    print(split, "images:", len(os.listdir(f"{extract_path}/{split}/images")))

## 3. Point `data.yaml` at the Colab folders

Class order matters: the application reads the class names from the model, so keep `With Helmet` first.

In [ ]:
yaml_content = """
train: /content/helmet_dataset/train/images
val: /content/helmet_dataset/valid/images
test: /content/helmet_dataset/test/images

nc: 2
names: ['With Helmet', 'Without Helmet']
"""

with open("/content/helmet_dataset/data.yaml", "w") as file:
    file.write(yaml_content)

print("Dataset configuration ready!")

## 4. Install Ultralytics

In [ ]:
!pip install -q ultralytics

## 5. Train (30 epochs, 640 px, batch 16)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")          # pretrained YOLO11 nano

model.train(
    data="/content/helmet_dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="/content/helmet_training",
    name="helmet_detector",
    exist_ok=True,
)

best_path = "/content/helmet_training/helmet_detector/weights/best.pt"
print("Best weights:", best_path, "(exists)" if os.path.exists(best_path) else "(NOT FOUND)")

## 6. Evaluate on the test split

In [ ]:
from ultralytics import YOLO

best = YOLO("/content/helmet_training/helmet_detector/weights/best.pt")
metrics = best.val(data="/content/helmet_dataset/data.yaml", split="test", imgsz=640, device=0)

print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"mAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")

## 7. Download the weights and the evaluation plots

In [ ]:
import shutil
from google.colab import files

files.download("/content/helmet_training/helmet_detector/weights/best.pt")

# evaluation plots (confusion matrix, PR curves, ...) from the validation run
shutil.make_archive("/content/helmet_evaluation", "zip", str(metrics.save_dir))
files.download("/content/helmet_evaluation.zip")